In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from pathlib import Path
import os

# 1. 경로 설정
BASE_DIR = Path(os.getcwd()).parent
PROCESSED_DIR = BASE_DIR / "01_data" / "02_processed"
HIST_PATH = PROCESSED_DIR / "01_sensor_timeslot_hist.csv"
SAVE_PATH = PROCESSED_DIR / "02_sensor_daily_features.csv"

# 2. 데이터 로드
print("📂 히스토그램 통합 데이터 로드 중...")
df = pd.read_csv(HIST_PATH)

def extract_features(group):
    """3단계 로직(지속성, 음압, 분포) 기반 Feature 추출 함수"""
    db_cols = [f"db_{i}" for i in range(100)]
    hist_matrix = group[db_cols].values # (6, 100) 행렬
    
    # 데이터 유효성 검사 (6개 타임슬롯 확인)
    if len(hist_matrix) < 6: return None
    
    # --- [1단계: 지속성(Consistency) Feature] ---
    # 6개 구간별 최빈값(Peak) 위치 추출
    peaks_per_slot = np.argmax(hist_matrix, axis=1)
    peak_std = round(np.std(peaks_per_slot), 2)
    
    # 6개 구간 간 평균 상관계수 (그래프 형태 유사도)
    corrs = []
    for i in range(5):
        # 두 구간의 데이터가 모두 0인 경우 대비 예외 처리
        if np.sum(hist_matrix[i]) > 0 and np.sum(hist_matrix[i+1]) > 0:
            corr, _ = pearsonr(hist_matrix[i], hist_matrix[i+1])
            if not np.isnan(corr): corrs.append(corr)
    avg_corr = round(np.mean(corrs), 3) if corrs else 0
    
    # --- 하루치 통합 데이터 생성 ---
    daily_hist = np.sum(hist_matrix, axis=0)
    total_count = np.sum(daily_hist)
    if total_count == 0: return None
    
    # --- [2단계: 음압(Amplitude) Feature] ---
    # 0을 제외한 최소 음압 (Lmin)
    nonzero_idx = np.where(daily_hist > 0)[0]
    critical_value = int(nonzero_idx.min()) if len(nonzero_idx) > 0 else 0
    
    # 일일 대표 최빈값 (Peak)
    daily_peak = int(np.argmax(daily_hist))
    
    # --- [3단계: 분포(Distribution) Feature] ---
    # 최빈값 주변(±3dB) 밀집도 (Peak Ratio)
    low_bound = max(0, daily_peak - 3)
    high_bound = min(99, daily_peak + 3)
    peak_area_count = np.sum(daily_hist[low_bound : high_bound + 1])
    peak_ratio = round((peak_area_count / total_count) * 100, 2)
    
    # 소음 발생 범위 (Spread)
    spread = int(nonzero_idx.max() - critical_value) if len(nonzero_idx) > 0 else 0

    return pd.Series({
        'avg_corr': avg_corr,          # [1차 핵심] 그래프 형태 유사도
        'peak_std': peak_std,          # [1차 보조] 봉우리 위치 일치도
        'critical_value': critical_value, # [2차 핵심] 최소 음압 (Lmin)
        'daily_peak': daily_peak,      # [2차 보조] 대표 소음 크기
        'peak_ratio': peak_ratio,      # [3차 핵심] 봉우리 밀집도 (%)
        'spread': spread,              # [3차 보조] 소음 범위
        'total_count': total_count     # 데이터 신뢰도
    })

# 3. 그룹화 및 저장
print("📊 3단계 로직 기반 일일 지표 산출 중...")
# date, site_name, sensor_id 기준으로 그룹화하여 지표 추출
daily_features = df.groupby(['date', 'site_name', 'sensor_id'], sort=False).apply(extract_features).reset_index()

# 결측치(None) 행 제거 후 저장
daily_features = daily_features.dropna()
daily_features.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')

print(f"✅ 분석용 Feature 파일 생성 완료: {SAVE_PATH}")
print(f"📈 생성된 데이터 규모: {len(daily_features):,} 행")


📂 히스토그램 통합 데이터 로드 중...
📊 3단계 로직 기반 일일 지표 산출 중...
✅ 분석용 Feature 파일 생성 완료: c:\workspace\wvr\classification\labeling_project\01_data\02_processed\sensor_daily_features.csv
📈 생성된 데이터 규모: 15,833 행
